# COVID-19 Global Data Analysis: Data Cleaning & Hygiene
## Part 2: Standardization, Missing Value Handling & Quality Validation

**Lead Data Engineer & Curator:** **Himanshu Bagde** ([GitHub: @Himanshubagde11](https://github.com/Himanshubagde11))  
**Primary Surveillance Source:** Our World in Data / WHO / Johns Hopkins CSSE  
**Project:** COVID-19 Global Data Analysis & Trend Visualization

This notebook documents the systematic data cleaning pipeline applied to the COVID-19 dataset.

### Cleaning Decisions Documented:
1. **Column Standardization:** Transformed all column headers to `snake_case`.
2. **Entity Normalization:** Standardized country names (e.g. US/USA -> United States, UK -> United Kingdom, Czechia -> Czech Republic) and isolated national entities from continental/income-group aggregates.
3. **Date Harmonization:** Parsed dates to strict ISO 8601 (`YYYY-MM-DD`).
4. **Deduplication:** Dropped conflicting duplicates on `(country, date)`, retaining latest observations.
5. **Flow Anomaly Smoothing:** Addressed retrospective negative reporting adjustments in `new_cases` and `new_deaths`.
6. **Domain-Specific Imputation:** Forward-filled cumulative metrics per country and filled flow NaNs with 0.



In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.ingestion.load_data import load_csv
from src.cleaning.clean_data import clean_covid_data
from src.validation.validate_data import run_data_quality_checks

DATA_PATH = ROOT_DIR / "data" / "raw" / "owid-covid-data.csv"
df_raw = load_csv(DATA_PATH)
print(f"Raw records loaded: {len(df_raw):,}")



### 1. Execute Production Cleaning Pipeline


In [ ]:
df_clean = clean_covid_data(df_raw, remove_aggregates=True, negative_strategy="zero")
print(f"Cleaned records: {len(df_clean):,}")
df_clean[['country', 'region', 'date', 'new_cases', 'total_cases', 'new_deaths', 'total_deaths']].head()



### 2. Verify Country Normalization


In [ ]:
sample_countries = ['United States', 'United Kingdom', 'South Korea', 'Czech Republic', 'Russia']
for c in sample_countries:
    count = (df_clean['country'] == c).sum()
    print(f"Canonical Country '{c}': {count:,} records found.")



### 3. Verify Absence of Negative Inflows


In [ ]:
neg_cases = (df_clean['new_cases'] < 0).sum()
neg_deaths = (df_clean['new_deaths'] < 0).sum()
print(f"Negative new_cases remaining: {neg_cases}")
print(f"Negative new_deaths remaining: {neg_deaths}")



### 4. Execute Automated Data Quality Assertion Suite


In [ ]:
dq_report = run_data_quality_checks(df_clean)
print(f"Overall DQ Status: [{dq_report['overall_status']}]")
for c in dq_report['checks']:
    print(f"- {c['name']}: {c['metric']} ({c['condition']}) -> [{c['status']}]")

